# STAC AI Workshop · NYC · 21 October 2026

## 03 · What does training change?

**Matthew Dixon, PhD, FRM · Participant workbook**

### Learning objectives
1. Compute SFT loss using a response-token mask
2. Explain LoRA within a selected projection
3. Compute DPO relative to a frozen reference
4. Separate a better objective from evidence support

Run the cells in order. Python 3.10+ and Jupyter are sufficient. All default exercises run offline. The figures and documents are synthetic unless a cell explicitly identifies recorded model outputs. No financial action is executed.

**Keep the companion `workshop_offline.py` and `data/` folder with these notebooks.**

## Open and run this notebook
[Open in Google Colab](https://colab.research.google.com/github/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/03_post_training.ipynb) · [View on GitHub](https://github.com/mfrdixon/STAC_AI_Bootcamp/blob/main/notebooks/03_post_training.ipynb)

1. Run the **COLAB_BOOTSTRAP** cell first. It downloads public helpers and saved fixtures when using Colab; it does nothing in local JupyterLab.
2. Run the named code cells from top to bottom. Leave live, fresh-inference and GPU switches off initially.
3. References such as **C03:5** mean line 5 inside code cell C03. The guide quotes the exact code so you can find it without editor line numbers. Enable line numbers in your editor preferences if available.
4. Read **Answer checks** below to compare the default result. Edited inputs and live runs can differ; the stated conditions explain what should remain true.

Saved results, constructed examples, rule checks and fresh model calls are identified separately. Do not save API keys in cells or outputs.

In [ ]:
# COLAB_BOOTSTRAP · download the public workshop helpers and saved fixtures.
import sys
if "google.colab" in sys.modules:
    from pathlib import Path
    import os, subprocess
    workshop_root = Path("/content/STAC_AI_Bootcamp")
    if not (workshop_root / ".git").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/mfrdixon/STAC_AI_Bootcamp.git", str(workshop_root)], check=True)
    os.chdir(workshop_root / "notebooks")
    sys.path.insert(0, str(workshop_root))
    sys.path.insert(0, str(workshop_root / "notebooks"))
    print("Workshop helpers and saved fixtures ready. No model API call was made.")

<!-- WORKSHOP-CODE-GUIDE C01 -->
### Code cell C01 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C01:4** | `notebook_dir=next((p for p in [Path.cwd(), Path.cwd()/"notebooks", *Path.cwd().parents] if (p/"workshop_offline.py").exists()),None)` | This is the explicit setup or input for the operation below. Inspect this value before changing it. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C01 · code cell
from pathlib import Path
import sys
notebook_dir=next((p for p in [Path.cwd(), Path.cwd()/"notebooks", *Path.cwd().parents] if (p/"workshop_offline.py").exists()),None)
if notebook_dir is None: raise RuntimeError("Extract the complete workshop ZIP and open this notebook inside its notebooks folder.")
if str(notebook_dir) not in sys.path: sys.path.insert(0,str(notebook_dir))
from workshop_offline import *
print("Offline workbook ready: no API key, Node.js, GPU or model download.")

## Same evidence as Web Section 3
Debt $75m, cash $15m, EBITDA $20m. **Signed covenant missing.** The reviewed answer requests the covenant rather than asserting comfortable headroom. All scores and tiny matrices below are declared illustrations; no model is trained.

## 1 · SFT: score the reviewed target, not the prompt
$L=-\sum_t m_t\log p(y_t\mid x,y_{<t})/\sum_t m_t$. Prompt mask 0; target mask 1. The three display units WITHHOLD / COVENANT / MISSING are illustrative tokens, not a claimed Qwen tokenization. Teacher forcing supplies the correct preceding answer tokens.

<!-- WORKSHOP-CODE-GUIDE C02 -->
### Code cell C02 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C02:3** | `initial=masked_nll([.9,.2,.1,.2],mask)` | Compute the selected target-token loss under the supplied illustrative probabilities. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C02 · code cell
mask=[0,1,1,1]
initial=masked_nll([.9,.2,.1,.2],mask)
improved=masked_nll([.9,.7,.6,.7],mask)
show_table([{"Scores":"Initial","Mean target loss":round(initial,3)},{"Scores":"Higher target likelihood","Mean target loss":round(improved,3)}])
assert masked_nll([.01,.2,.1,.2],mask)==initial
assert improved<initial
print("The prompt score changes no loss because its mask is zero.")

## 2 · LoRA: parallel trainable branch
$h=Wx+(\alpha/r)BAx$. W is frozen; A and B are trained using the chosen objective. LoRA adds parameters inside selected projections, not a new model between blocks.

<!-- WORKSHOP-CODE-GUIDE C03 -->
### Code cell C03 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C03:2** | `show_table([{"Rank":r,"A dimensions":f"{r} × 896","B dimensions":f"896 × {r}","Added parameters":lora_count(896,896,r)} for r in (4,8,16)])` | Count adapter parameters from the dimensions and rank. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C03 · code cell
show_table([{"Rank":r,"A dimensions":f"{r} × 896","B dimensions":f"896 × {r}","Added parameters":lora_count(896,896,r)} for r in (4,8,16)])
x=[2,1]; A=[1,-1]; B=[.5,-.5]; scale=2
base=x[:]  # W is identity
ax=sum(a*v for a,v in zip(A,x)); correction=[scale*b*ax for b in B]
output=[v+d for v,d in zip(base,correction)]
show_table([{"Quantity":"W x","Value":str(base)},{"Quantity":"A x","Value":ax},{"Quantity":"Scaled correction","Value":str(correction)},{"Quantity":"Output","Value":str(output)}])
assert output==[3,0]
base_count=494032768; adapter_count=(lora_count(896,896,8)+lora_count(896,128,8))*24
assert adapter_count==540672
print(f"Recorded earlier run: {base_count:,} + {adapter_count:,} = {base_count+adapter_count:,} parameters.")
print("The tiny matrices above were supplied, not fitted. Actual adapter size is a separate recorded fact.")

## 3 · DPO: same prompt, preferred versus rejected response
Chosen: “Withhold; obtain the signed covenant.” Rejected: “Comfortable headroom; release.”

$z=\beta[(\log\pi(y^+)-\log\pi(y^-))-(\log\pi_{ref}(y^+)-\log\pi_{ref}(y^-))]$ and $L=-\log\sigma(z)$.
Scores are whole-response summed log probabilities. The reference remains (−8, −6), beta 0.1.

<!-- WORKSHOP-CODE-GUIDE C04 -->
### Code cell C04 · exact lines to inspect

| Line | Code | Meaning |
|---|---|---|
| **C04:4** | `value=dpo(c,r)` | Compute the illustrated preference loss relative to the fixed reference scores. |

Run this cell, then compare its output with the adjacent explanation and the **Answer checks** at the end. Line numbers restart in each named code cell; execution counters are not line numbers.

In [ ]:
# C04 · code cell
rows=[]
for label,c,r in [("Same as reference",-8,-6),("Favour chosen response",-5,-8),("Favour rejected response",-9,-5)]:
    value=dpo(c,r)
    rows.append({"Policy":label,"Chosen score":c,"Rejected score":r,"Margin":round(value["margin"],3),"DPO loss":round(value["loss"],3)})
show_table(rows)
assert dpo(-5,-8)["loss"]<dpo(-8,-6)["loss"]<dpo(-9,-5)["loss"]

## 4 · Inspect evidence support
**Missing covenant → definition/ceiling unresolved → claim unsupported.** None of the arithmetic above changes that fact. Training on a preference is not an execution permission or calibrated probability of truth.

**Check:** what did lower loss establish? Higher likelihood of the declared target or improved agreement with the chosen preference—not factual correctness, generalization or alpha.

## Main takeaway
SFT/DPO choose what to reward; LoRA chooses how weights can change. The evidence graph still requires the missing document. Actual adaptation and historical replay are separate Section 4 exercises.

**Exit ticket:** explain one result, the evidence for it, and a limit on the conclusion.

## Answer checks · default configuration

- The illustrated masked mean loss changes from approximately **1.840 to 0.408**. Changing the masked-out prompt probability does not change the loss. These are supplied probabilities, not a new model-training run.
- For a single 896×896 target: rank 4 adds **7,168**, rank 8 **14,336**, rank 16 **28,672** parameters.
- Tiny LoRA arithmetic: base `[2, 1]`, scaled correction `[1, -1]`, output **`[3, 0]`**. The matrices are supplied, not fitted in this cell.
- The earlier recorded adapter has **540,672** parameters over a **494,032,768**-parameter base; this notebook does not shrink or retrain the base.
- DPO losses: same as reference **0.693**, favor chosen **0.474**, favor rejected **0.798** under the declared example scores and beta.
- Better preference loss does not supply a missing covenant or validate truth probabilities. SFT/DPO choose the objective; LoRA chooses the trainable update.

If your result differs, check which input, selected case, data version or optional switch changed. For a live model, inspect actual tool calls and returned evidence; do not assume the recorded result is guaranteed.